# Post-hoc GNNExplainer on Vanilla 4-layer GINE on Di-Halo_Benzene

This notebook trains a vanilla 4-layer **GINE** classifier on your custom BA2Motif dataset (with ground-truth `node_mask`), then fits a **post-hoc GNNExplainer** and evaluates **Jaccard@|GT|** and **Node AUROC**.

Dataset file used: `/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data/processed/data.pt`

In [19]:
import os
import sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from copy import deepcopy
from torch.nn import Sequential, Linear, ReLU, BatchNorm1d
from torch_geometric.data import InMemoryDataset, Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_add_pool
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

# -------------------------
# Repro / Device
# -------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# -------------------------
# Ensure repo root is on sys.path for bcosgnn imports
# -------------------------
current_dir = Path.cwd()
def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    return start # Fallback

project_root = find_repo_root(current_dir)
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")

DEVICE = cpu
Repo root added: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn


In [9]:
class ZincProcessedDataset(InMemoryDataset):
    def __init__(self, root, transform=None, pre_transform=None):
        super().__init__(root, transform, pre_transform)
        data_path = Path(self.processed_dir) / "data.pt"
        try:
            self.data, self.slices = torch.load(data_path, weights_only=False)
        except TypeError: 
            self.data, self.slices = torch.load(data_path)

    @property
    def processed_file_names(self):
        return ['data.pt']

    def process(self):
        pass

# Locate Dataset
possible_paths = [
    "multi_class_Zinc/zinc_di_halo_benzene_data",
    "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data"
]
dataset_path = None
for rel_path in possible_paths:
    full_path = project_root / rel_path
    if full_path.exists():
        dataset_path = str(full_path.resolve())
        break
if dataset_path is None:
    dataset_path = str((project_root / "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data").resolve())

print(f"Loading dataset from: {dataset_path}")
dataset = ZincProcessedDataset(root=dataset_path)

# Debug: Just print available keys to be sure, but DO NOT patch.
print("First graph keys:", dataset[0].keys)

Loading dataset from: /Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data
First graph keys: <bound method BaseData.keys of Data(x=[51, 10], edge_index=[2, 110], edge_attr=[110, 4], y=[1], explanation_mask=[51])>


In [20]:
def get_ground_truth_mask(data):
    """Finds GT mask from 'explanation_mask', 'node_mask', etc."""
    keys_to_check = ['node_mask', 'explanation_mask', 'explaination_mask', 'explanation']
    for key in keys_to_check:
        if hasattr(data, key):
            mask = getattr(data, key)
            if mask is not None:
                return mask
    return None

def evaluate_custom_jaccard(explainer, model, dataset):
    jaccard_scores = []
    print(f"Evaluating Jaccard on {len(dataset)} graphs...")
    
    for data in tqdm(dataset):
        data = data.to(DEVICE)
        
        # 1. Get GT
        gt_mask = get_ground_truth_mask(data)
        if gt_mask is None: continue
        
        gt_mask = gt_mask.squeeze().cpu().numpy()
        gt_nodes = set(np.where(gt_mask == 1)[0])
        k = len(gt_nodes)
        if k == 0: continue

        # 2. Get Pred
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_attr, batch=None)
            target = logits.argmax().item()
            
        explanation = explainer(
            data.x, 
            data.edge_index, 
            target=torch.tensor([target], device=DEVICE),
            edge_attr=data.edge_attr
        )
        
        # 3. Score (FIXED FLATTENING)
        # Flatten ensures we get a 1D array of scores, so argsort returns simple integers
        pred_mask = explanation.node_mask.detach().cpu().numpy().flatten()
        
        top_k_indices = np.argsort(pred_mask)[-k:]
        pred_nodes = set(top_k_indices)

        intersection = len(gt_nodes.intersection(pred_nodes))
        union = len(gt_nodes.union(pred_nodes))
        jaccard_scores.append(intersection / union)

    if not jaccard_scores: return 0.0
    return np.mean(jaccard_scores)

def evaluate_custom_auroc(explainer, model, dataset):
    auroc_scores = []
    print(f"Evaluating AUROC on {len(dataset)} graphs...")
    
    for data in tqdm(dataset):
        data = data.to(DEVICE)
        
        gt_mask = get_ground_truth_mask(data)
        if gt_mask is None: continue
        gt_mask = gt_mask.squeeze().cpu().numpy()
        
        if gt_mask.sum() == 0 or gt_mask.sum() == len(gt_mask):
            continue

        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_attr, batch=None)
            target = logits.argmax().item()

        explanation = explainer(
            data.x, 
            data.edge_index, 
            target=torch.tensor([target], device=DEVICE),
            edge_attr=data.edge_attr
        )
        
        # FIX: Flatten here too for consistency
        pred_mask = explanation.node_mask.detach().cpu().numpy().flatten()
        
        try:
            score = roc_auc_score(gt_mask, pred_mask)
            auroc_scores.append(score)
        except ValueError: pass

    if not auroc_scores: return 0.0
    return np.mean(auroc_scores)

In [11]:
class VanillaGINE4(nn.Module):
    def __init__(self, in_dim, edge_dim, hidden_dim, num_classes, drop_ratio=0.5):
        super().__init__()
        self.node_proj = Linear(in_dim, hidden_dim)
        self.edge_proj = Linear(edge_dim, hidden_dim)

        def mlp():
            return Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim),
            )

        self.conv1 = GINEConv(nn=mlp(), train_eps=True)
        self.bn1 = BatchNorm1d(hidden_dim)
        self.conv2 = GINEConv(nn=mlp(), train_eps=True)
        self.bn2 = BatchNorm1d(hidden_dim)
        self.conv3 = GINEConv(nn=mlp(), train_eps=True)
        self.bn3 = BatchNorm1d(hidden_dim)
        self.conv4 = GINEConv(nn=mlp(), train_eps=True)
        self.bn4 = BatchNorm1d(hidden_dim)

        self.drop_ratio = drop_ratio
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr, batch=None):
        if batch is None: batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        
        h = self.node_proj(x.float())
        edge_emb = self.edge_proj(edge_attr.float())

        h = F.relu(self.bn1(self.conv1(h, edge_index, edge_attr=edge_emb)))
        h = F.relu(self.bn2(self.conv2(h, edge_index, edge_attr=edge_emb)))
        h = F.relu(self.bn3(self.conv3(h, edge_index, edge_attr=edge_emb)))
        h = F.relu(self.bn4(self.conv4(h, edge_index, edge_attr=edge_emb)))

        hg = global_add_pool(h, batch)
        hg = F.dropout(hg, p=self.drop_ratio, training=self.training)
        return self.classifier(hg)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss = criterion(logits, batch.y.view(-1).long())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total_correct = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        pred = logits.argmax(dim=-1)
        total_correct += (pred == batch.y.view(-1).long()).sum().item()
    return total_correct / len(loader.dataset)

def get_gnn_explainer(model):
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=200, lr=0.01),
        explanation_type='model',
        node_mask_type='object',  
        edge_mask_type=None,      
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

In [21]:

print(f"Running Experiment for SEED {SEED}...")

# Split Data
indices = np.arange(len(dataset))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n = len(indices)
train_dataset = dataset[indices[:int(0.8*n)]]
val_dataset = dataset[indices[int(0.8*n):int(0.9*n)]]
test_dataset = dataset[indices[int(0.9*n):]]

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Init Model
model = VanillaGINE4(
    in_dim=dataset.num_features, 
    edge_dim=dataset.num_edge_features, 
    hidden_dim=64, 
    num_classes=dataset.num_classes
).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# Train Loop
best_acc = 0.0
patience = 0
best_state = None

for epoch in range(1, 101):
    loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_acc = eval_one_epoch(model, val_loader, criterion)
    scheduler.step(loss)
    
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = deepcopy(model.state_dict())
        patience = 0
    else:
        patience += 1
        
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: Loss {loss:.4f} | Val Acc {val_acc:.4f}")
        
    if patience >= 25:
        print(f"Early stopping at epoch {epoch}.")
        break

if best_state:
    model.load_state_dict(best_state)

test_acc = eval_one_epoch(model, test_loader, criterion)
print(f"Test Accuracy: {test_acc:.4f}")

# Explain
print("Evaluating Explainer...")
gnn_explainer = get_gnn_explainer(model)

# Use CUSTOM evaluation functions (No patching needed)
jacc = evaluate_custom_jaccard(gnn_explainer, model, test_dataset)
auc = evaluate_custom_auroc(gnn_explainer, model, test_dataset)

print(f"\nFinal Results (Seed {SEED}):")
print(f"Jaccard: {jacc:.4f}")
print(f"AUROC:   {auc:.4f}")

Running Experiment for SEED 42...
Epoch 10: Loss 0.0033 | Val Acc 1.0000
Epoch 10: Loss 0.0033 | Val Acc 1.0000
Epoch 20: Loss 0.0006 | Val Acc 1.0000
Epoch 20: Loss 0.0006 | Val Acc 1.0000
Early stopping at epoch 27.
Test Accuracy: 1.0000
Evaluating Explainer...
Evaluating Jaccard on 900 graphs...
Early stopping at epoch 27.
Test Accuracy: 1.0000
Evaluating Explainer...
Evaluating Jaccard on 900 graphs...


  0%|          | 0/900 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:32: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:32: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
100%|██████████| 900/900 [05:00<00:00,  2.99it/s]


Evaluating AUROC on 900 graphs...


  0%|          | 0/900 [00:00<?, ?it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 1/900 [00:00<04:55,  3.05it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:71: UserWarning: The 'target' should not be provided for the explanation type 'model'
  explanation = explainer(
  0%|          | 2/900 [00:00<05:12,  2.87it/s]/var/folders/7z/v5vlp84n7pl7lqlv_7xw5_z40000gn/T/ipykernel_4608/1759294647.py:71: UserWarning: The 'tar


Final Results (Seed 42):
Jaccard: 0.3569
AUROC:   0.6852


In [8]:
print("\n--- PATCHING DATASET ---")
sample_data = dataset[0]
potential_keys = ['explanation_mask', 'explaination_mask', 'explanation']
found_key = None

for key in potential_keys:
    if key in sample_data:
        found_key = key
        break

if found_key:
    print(f"Found ground truth in '{found_key}'. Copying to 'node_mask'...")
    for data in dataset:
        data.node_mask = getattr(data, found_key)
else:
    print("!! WARNING !! Could not find any explanation mask attribute.")

# Verification
if hasattr(dataset[0], 'node_mask'):
    print(f"Success! node_mask shape: {dataset[0].node_mask.shape}")
else:
    print("Failure: node_mask is still missing.")
print("------------------------\n")


--- PATCHING DATASET ---
Found ground truth in 'explanation_mask'. Copying to 'node_mask'...
Failure: node_mask is still missing.
------------------------



In [3]:
class VanillaGINE4(nn.Module):
    def __init__(
        self, 
        in_dim: int,      # Node feature dim (9)
        edge_dim: int,    # Edge feature dim (4)
        hidden_dim: int, 
        num_classes: int, 
        drop_ratio: float = 0.5
    ):
        super().__init__()
        
        # 1. Project Node Features
        self.node_proj = Linear(in_dim, hidden_dim)
        
        # 2. Project Edge Features (Required for GINE)
        self.edge_proj = Linear(edge_dim, hidden_dim)

        def mlp():
            return Sequential(
                Linear(hidden_dim, 2 * hidden_dim),
                BatchNorm1d(2 * hidden_dim),
                ReLU(),
                Linear(2 * hidden_dim, hidden_dim),
            )

        # 3. GINE Layers
        self.conv1 = GINEConv(nn=mlp(), train_eps=True)
        self.bn1 = BatchNorm1d(hidden_dim)
        
        self.conv2 = GINEConv(nn=mlp(), train_eps=True)
        self.bn2 = BatchNorm1d(hidden_dim)
        
        self.conv3 = GINEConv(nn=mlp(), train_eps=True)
        self.bn3 = BatchNorm1d(hidden_dim)
        
        self.conv4 = GINEConv(nn=mlp(), train_eps=True)
        self.bn4 = BatchNorm1d(hidden_dim)

        self.drop_ratio = drop_ratio
        self.classifier = Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, edge_attr, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)

        # Project inputs
        h = self.node_proj(x.float())
        edge_emb = self.edge_proj(edge_attr.float()) # Map 4 dims -> 64 dims

        # Message Passing (Passing edge_emb to every layer)
        h = self.conv1(h, edge_index, edge_attr=edge_emb)
        h = self.bn1(h)
        h = F.relu(h)
        
        h = self.conv2(h, edge_index, edge_attr=edge_emb)
        h = self.bn2(h)
        h = F.relu(h)
        
        h = self.conv3(h, edge_index, edge_attr=edge_emb)
        h = self.bn3(h)
        h = F.relu(h)
        
        h = self.conv4(h, edge_index, edge_attr=edge_emb)
        h = self.bn4(h)
        h = F.relu(h)

        # Readout
        hg = global_add_pool(h, batch)
        hg = F.dropout(hg, p=self.drop_ratio, training=self.training)
        
        return self.classifier(hg)

In [4]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        if batch.num_nodes <= 1: continue

        # Pass edge_attr for GINE
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        y = batch.y.view(-1).to(torch.long)

        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * int(batch.num_graphs)
        pred = logits.argmax(dim=-1)
        total_correct += int((pred == y).sum().item())
        total_graphs += int(batch.num_graphs)

    return (total_loss / max(total_graphs, 1)), (total_correct / max(total_graphs, 1))

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(DEVICE)
        if batch.num_nodes <= 1: continue

        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        y = batch.y.view(-1).to(torch.long)
        loss = criterion(logits, y)

        total_loss += float(loss.item()) * int(batch.num_graphs)
        pred = logits.argmax(dim=-1)
        total_correct += int((pred == y).sum().item())
        total_graphs += int(batch.num_graphs)

    return (total_loss / max(total_graphs, 1)), (total_correct / max(total_graphs, 1))

def get_gnn_explainer(model, epochs: int = 200, lr: float = 0.01):
    # Node mask 'object' works with standard GINE without custom weighted layers.
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=epochs, lr=lr),
        explanation_type='model',
        node_mask_type='object', # Learn atom importance
        edge_mask_type=None,     # Skip edge masks for standard GINE
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )

In [5]:
from bcosgnn.evaluation import evaluate_gnnexplainer_jaccard, evaluate_gnnexplainer_auroc

SEEDS = [0, 1, 2, 3, 4]
results_jaccard = []
results_auroc = []
results_test_acc = []

in_dim = dataset.num_features       # 9
edge_dim = dataset.num_edge_features # 4
num_classes = dataset.num_classes

for seed in SEEDS:
    print(f"\n{'='*20} SEED {seed} {'='*20}")
    
    # 1. Set Seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 2. Split Data
    indices = np.arange(len(dataset_list))
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    
    n = len(indices)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    
    train_dataset = [dataset_list[i] for i in indices[:n_train]]
    val_dataset = [dataset_list[i] for i in indices[n_train:n_train + n_val]]
    test_dataset = [dataset_list[i] for i in indices[n_train + n_val:]]
    
    BATCH_SIZE = 128
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    # 3. Initialize Vanilla GINE
    model = VanillaGINE4(
        in_dim=in_dim, 
        edge_dim=edge_dim, 
        hidden_dim=64, 
        num_classes=num_classes, 
        drop_ratio=0.5
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = torch.nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10, min_lr=1e-6
    )
    
    # 4. Train
    EPOCHS = 100
    EARLY_STOP_PATIENCE = 25
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epoch % 50 == 0:
             print(f"  Epoch {epoch:03d}: val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch}")
            break
            
    if best_state is not None:
        model.load_state_dict(best_state)
    
    test_loss, test_acc = eval_one_epoch(model, test_loader, criterion)
    results_test_acc.append(test_acc)
    print(f"  Best Val Loss: {best_val_loss:.4f} | Test Acc: {test_acc:.4f}")
    
    # 5. Explainer Evaluation
    print("  Evaluating Explainer...")
    gnn_explainer = get_gnn_explainer(model, epochs=200, lr=0.01)
    
    # Pass 'edge_attr' implicitly via dataset items
    jacc = evaluate_gnnexplainer_jaccard(gnn_explainer, model, test_dataset)
    auc = evaluate_gnnexplainer_auroc(gnn_explainer, model, test_dataset)
    
    results_jaccard.append(jacc)
    results_auroc.append(auc)
    print(f"  Seed {seed} Result -> Jaccard: {jacc:.4f}, AUROC: {auc:.4f}")

# 6. Report Final Stats
print("\n" + "#"*40)
print("FINAL RESULTS (Mean \u00B1 Std)")
print("#"*40)
print(f"Test Acc:      {np.mean(results_test_acc):.4f} \u00B1 {np.std(results_test_acc):.4f}")
print(f"Jaccard@|GT|:  {np.mean(results_jaccard):.4f} \u00B1 {np.std(results_jaccard):.4f}")
print(f"Node AUROC:    {np.mean(results_auroc):.4f} \u00B1 {np.std(results_auroc):.4f}")


==================== SEED 0 ====================
  Early stopping at epoch 28
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...
  Early stopping at epoch 28
  Best Val Loss: 0.0000 | Test Acc: 1.0000
  Evaluating Explainer...


Evaluating AUROC (gnnexplainer): 100%|██████████| 900/900 [00:00<00:00, 42808.24it/s]

  Seed 0 Result -> Jaccard: nan, AUROC: nan

==================== SEED 1 ====================


KeyboardInterrupt: 